<a href="https://colab.research.google.com/github/hparupudi/AIVideoEditor/blob/main/AIVideoEditor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Necessary Installations
!pip install git+https://github.com/openai/CLIP.git

In [ ]:
import requests
from openai import OpenAI
from google.colab import userdata
import json

#OpenAI setup
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

#Generates keyword search term from given script segment
def id_keyword(script):
  DEVELOPER_PROMPT = '''You are a documentary visual researcher. For each script segment, you choose the stock footage search term an experienced editor would type to find the most specific, recognizable shot for that moment.

    ## How to choose the term
    1. Find the concrete subject. Abstract or rhetorical lines still refer to something concrete. Use the video topic to identify the real organizations, products, places, people, species, or events being discussed, and show those.
    2. Name it when you can. Prefer proper nouns and specific types over broad categories:
      - "Tesla Gigafactory Berlin", not "electric car factory"
      - "Great Barrier Reef bleached coral", not "ocean reef"
      - "Colosseum aerial Rome", not "ancient ruins"
      If the segment or topic names an entity, use it. If they only imply a category, pick the most representative real-world example of that category.
    3. If no named entity fits, be precise instead. Some topics have no relevant brand, place, or person. Then describe the exact subject and action rather than a vague category: "runner tying shoelaces", not "fitness"; "brain MRI scan monitor", not "medical research".
    4. Make it visual. The term must describe something a camera can film: an object, a place, an organism, or a person doing something. Never output concepts like "future", "economy", "innovation", "health", or "growth".
    5. Add one visual detail when it helps: an action (launching, harvesting, typing), a setting (laboratory, stadium, trading floor), a time or condition (at night, in snow), or a shot type (aerial, close-up, timelapse).
    6. Keep the term 2 to 5 words, with no punctuation.

    ## Output
    Return only a JSON object:
    {"term": "<search term>"}

    The term must contain a proper noun or specific type whenever the segment or topic supports one.

    ## Examples
    Topic: Nvidia's dominance in AI chips
    Segment: "Investors didn't see it coming."
    {"term": "NASDAQ trading floor"}

    Topic: Climate change and coral reef decline
    Segment: "Within a decade, half of it could be gone."
    {"term": "Great Barrier Reef bleached coral"}

    Topic: The fall of the Roman Empire
    Segment: "It was the greatest power the world had ever known."
    {"term": "Colosseum aerial Rome"}

    Topic: SpaceX and the commercial space race
    Segment: "The cost of reaching orbit has collapsed."
    {"term": "Falcon 9 booster landing"}

    Topic: How sleep affects memory
    Segment: "Most of us aren't getting nearly enough of it."
    {"term": "man awake in bed at night"}'''

  completion = client.chat.completions.create(
      model="gpt-6.1-sol",
      response_format={'type': 'json_object'},
      messages = [
          {'role': 'developer', 'content': DEVELOPER_PROMPT},
          {'role': 'user', 'content': f'{script}'}
      ]
  )
  search_term = json.loads(completion.choices[0].message.content)["term"]
  min_duration = round((script.count(" ") + 1) / 2, 1)
  max_duration = 90

  return search_term, min_duration, max_duration


#Defines global parameters for Vecteezy API requests
user_id = userdata.get("Vecteezy_ID")
api_key = userdata.get("Vecteezy_API")
headers = {
      'accept': 'application/json',
      'Authorization': f'Bearer {api_key}'
  }

#Finds most relevant stock footage videos based on given keyword
def find_vids(keyword, min_duration, max_duration):
  search_keyword = keyword.replace(' ', '%20')
  url = f"https://api.vecteezy.com/v2/{user_id}/resources?term={search_keyword}&content_type=video&page=1&per_page=5&sort_by=relevance&license_type=commercial&ai_generated=false&duration={min_duration}_{max_duration}"
  result = requests.get(url, headers=headers).json()
  vid_data = {item['title']: item['id'] for item in result['resources']}
  return keyword, vid_data

#keyword, vid_data = find_vids('Boeing 777 Taxiing', 10, 90)

In [ ]:
from openai import OpenAI
import urllib.request

#Identifies video that best matches keyword & fetches downloadable link
def fetch_top_vid(keyword, vid_data):
  client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
  vid_titles = [title for title in vid_data.keys()]
  completion = client.chat.completions.create(
      model="gpt-6-luna",
      messages=[
          {'role': 'developer', 'content': 'Given a search term and a list of video titles, return the video title that most closely matches the search term.'},
          {'role': 'user', 'content': f'Search term: {keyword}, Video titles: {vid_titles}'}
      ]
  )
  top_vid = completion.choices[0].message.content
  vid_id = vid_data[top_vid]
  url = f"https://api.vecteezy.com/v2/{user_id}/resources/{vid_id}/download"
  download_link = requests.get(url, headers=headers).json()['url']
  return download_link

#Downloads video to Google Colab
def download_vid(download_link, video_path):
  response = requests.get(download_link, stream=True)
  with open(video_path, "wb") as file:
    for chunk in response.iter_content(chunk_size = 1024 ** 2):
      if chunk:
        file.write(chunk)
#download_link = fetch_top_vid(keyword, vid_data)

In [ ]:
import cv2
from PIL import Image
import torch, clip

#Saves one frame per second in an array
def save_frames(video_path):
  vid = cv2.VideoCapture(video_path)
  fps = round(vid.get(cv2.CAP_PROP_FPS), 0)
  frame_array, frame_idx = [], 0
  while vid.isOpened():
    ret, frame = vid.read()
    if not ret: break
    if frame_idx % fps == 0:
      rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
      frame_array.append(rgb_frame)
    frame_idx += 1
  vid.release()
  return frame_array

#Uses CLIP embedding to find optimal timestamps of video given text
def get_timestamps(script, frame_arr, duration):
  device = "cuda" if torch.cuda.is_available() else "cpu"
  model, preprocess = clip.load("ViT-B/32", device=device)
  frames = [preprocess(Image.fromarray(f)) for f in frame_arr]
  with torch.no_grad():
    img = model.encode_image(torch.stack(frames).to(device))
    txt = model.encode_text(clip.tokenize([script]).to(device))

  #print(f"Img shape before norm: {img.shape}")
  #print(f"Text shape before norm: {txt.shape}")
  img = img / img.norm(dim=-1, keepdim=True)
  txt = txt / txt.norm(dim=-1, keepdim=True)
  #print(f"Img shape after norm: {img.shape}")
  #print(f"Text shape after norm: {txt.shape}")
  scores = (img @txt.T).squeeze()
  #print(f"Scores shape: {scores.shape}")

  window = scores.unfold(0, duration, 1).mean(1)
  start = window.argmax().item()
  return start


In [ ]:
#Full pipeline
import math

def find_stock_footage(script):
  search_term, min_duration, max_duration = id_keyword(script)
  duration = math.ceil(min_duration)
  keyword, vid_data = find_vids(search_term, duration, max_duration)
  download_link = fetch_top_vid(keyword, vid_data)
  video_path = "video.mp4"
  download_vid(download_link, video_path)
  frame_arr = save_frames(video_path)
  start = get_timestamps(script, frame_arr, duration)
  return start, start + min_duration

script = "What does it mean for the future of ultra low-cost commercial aviation?"
start, end = find_stock_footage(script)
print(start, end)

6 12.0


In [ ]:
script = "What does it mean for the future of ultra low-cost commercial aviation?"
search_term, min_duration, max_duration = id_keyword(script)
search_term

'Ryanair Boeing 737 takeoff'